In [ ]:
%pip install requests beautifulsoup4 


In [ ]:

import re
import requests
from bs4 import BeautifulSoup
from pathlib import Path

In [3]:
def get_books_and_links(toc_url):
    response = requests.get(toc_url)
    soup = BeautifulSoup(response.text, 'html.parser')

    books = {}
    current_book = None

    tags = soup.find_all(["h2", "ul"])
    i = 0
    while i < len(tags):
        tag = tags[i]

        if tag.name == "h2" and "Book" in tag.get_text():
            current_book = tag.get_text(strip=True)
            books[current_book] = []

            if i + 1 < len(tags) and tags[i + 1].name == "ul":
                ul = tags[i + 1]

                chapter_lis = ul.find_all("li", recursive=False)
                if len(chapter_lis) == 1 and chapter_lis[0].find("ul"):
                    chapter_lis = chapter_lis[0].find("ul").find_all("li", recursive=False)

                for li in chapter_lis:
                    a = li.find("a", href=True)
                    if a:
                        title = a.get_text(strip=True)
                        href = a["href"]
                        books[current_book].append((title, href))

                i += 1  # Skip the UL

        i += 1

    return books

In [4]:
def sanitize_filename(title, index):
    title = re.sub(r'[^\w\s-]', '', title)
    title = re.sub(r'\s+', '_', title.strip())
    return f"Chapter_{index:02d}_{title}.pdf"

In [ ]:
def save_books_as_pdfs(books, output_dir=".", wkhtmltopdf_path=None):
    if wkhtmltopdf_path:
        config = pdfkit.configuration(wkhtmltopdf=wkhtmltopdf_path)
    else:
        config = None

    output_dir = Path(output_dir)

    options = {
        'quiet': '',
        'no-print-media-type': '',
        'disable-smart-shrinking': '',
        'disable-javascript': '',
        'load-error-handling': 'ignore',
        'load-media-error-handling': 'ignore'
    }

    for book_title, chapters in books.items():
        print(f"\n📘 Saving {book_title}...")

        book_folder = output_dir / book_title.replace(" ", "_")
        book_folder.mkdir(exist_ok=True)

        for i, (chapter_title, url) in enumerate(chapters, start=1):
            file_name = sanitize_filename(chapter_title, i)
            file_path = book_folder / file_name
            print(f"  - Saving: {chapter_title} → {file_name}")

            try:
                pdfkit.from_url(url, str(file_path), configuration=config, options=options)
            except Exception as e:
                print(f"    ⚠️ Failed to save {chapter_title}: {e}")

In [7]:
import shutil
print(shutil.which("wkhtmltopdf"))

None


In [ ]:
# Set this if wkhtmltopdf is not in PATH:
wkhtmltopdf_path = "C:/Program Files/wkhtmltox/bin/wkhtmltopdf.exe"

import shutil
print(shutil.which("wkhtmltopdf"))
toc_url = "https://practicalguidetoevil.wordpress.com/table-of-contents/"
books = get_books_and_links(toc_url)

# download book by book
books = {k: v for k, v in books.items() if k == "Book 7"}

save_books_as_pdfs(books, output_dir=".", wkhtmltopdf_path=wkhtmltopdf_path)

None

📘 Saving Book 6...
  - Saving: Prologue → Chapter_01_Prologue.pdf
  - Saving: Chapter 1: Recommence → Chapter_02_Chapter_1_Recommence.pdf
  - Saving: Chapter 2: Enlistment → Chapter_03_Chapter_2_Enlistment.pdf
  - Saving: Chapter 3: Standard → Chapter_04_Chapter_3_Standard.pdf
  - Saving: Chapter 4: Shadowed → Chapter_05_Chapter_4_Shadowed.pdf
  - Saving: Chapter 5: Expired → Chapter_06_Chapter_5_Expired.pdf
  - Saving: Chapter 6: Equivalent → Chapter_07_Chapter_6_Equivalent.pdf
  - Saving: Chapter 7: Approach → Chapter_08_Chapter_7_Approach.pdf
  - Saving: Chapter 8: Stanchion → Chapter_09_Chapter_8_Stanchion.pdf
  - Saving: Chapter 9: Acceleration → Chapter_10_Chapter_9_Acceleration.pdf
  - Saving: Chapter 10:Reflections → Chapter_11_Chapter_10Reflections.pdf
  - Saving: Chapter 11: Veer → Chapter_12_Chapter_11_Veer.pdf
  - Saving: Chapter 12: Contest → Chapter_13_Chapter_12_Contest.pdf
  - Saving: Interlude: Truce → Chapter_14_Interlude_Truce.pdf
  - Saving: Interlude: Terms →